# Udemy Purchase Scraper

In [ ]:
from __future__ import annotations

from undetected_chromedriver import By

from dotenv import load_dotenv, find_dotenv
import undetected_chromedriver as uc
import os

import time
import json
import re
import requests

import pickle
from copy import deepcopy

: 

In [ ]:
import udemy_scraper as us
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Start driver
driver = us.browser.startBrowser()

AttributeError: module 'src.udemy_scraper' has no attribute 'browser'

Manually log into Udemy now because trying to log in automatically with 2FA and everything will be a pain in the ass. Save the cookies to skip this step later (and prevent being locked out if you log in too many times).

## Loading

In [ ]:
driver.get("https://www.udemy.com/")
time.sleep(int(os.environ.get("SLEEP_TIME", 3)))

In [ ]:
cookies_path = os.environ.get("COOKIES_PATH", "cookies.json")
saveCookies(driver=driver, f_path=cookies_path, force_cookies=force_new_cookies)
loadCookies(driver=driver, f_path=cookies_path)
time.sleep(int(os.environ.get("SLEEP_TIME", 3)))

## Get Purchases

In [ ]:
purchases = getPurchasePages(driver=driver)

In [ ]:
backup_purc = deepcopy(purchases)

In [ ]:
unique_courses = {}
for purchase in purchases:
    purchase.courses = [course for course in purchase.courses if course.course_link not in unique_courses]
    for course in purchase.courses:
        if course.course_link not in unique_courses:
            unique_courses[course.course_link] = None
del unique_courses

## Get Ownership

In [ ]:
# If I had any archived courses I would also want to check those. Should be the same functions and then just extend the lists.
owned_courses = getOwnedPages(driver)

In [ ]:
unique_courses = {}
# new_courses = []
# for course in owned_courses:
#     if course.course_link not in unique_courses:
#         unique_courses[course.course_link] = None
#         new_courses.append(course)
ind = 0
while ind < len(owned_courses):
    c_link = owned_courses[ind].course_link
    if(c_link not in unique_courses):
        unique_courses[c_link] = None
        ind = ind + 1
    else:
        del owned_courses[ind]
del unique_courses

In [ ]:
backup_owned = deepcopy(owned_courses)

## Get Owned Real URL

In [ ]:
requests_cookies = requestsCookiesFromSelenium(cookies_file=cookies_path)

## Get URLS

In [ ]:
with requests.Session() as session:
    #session.headers.update(requests_cookies)
    session.cookies = requests_cookies #requests.utils.dict_from_cookiejar(jar)
    for ind, course in enumerate(owned_courses):
        owned_courses[ind].course_page = f"{getOwnedCurrentURLRequest(session=session, course_link=course.course_link)}/"

## Temporary Data Serialization

In [ ]:
data_save = {"purchases": purchases, "owned_courses": owned_courses}

with open("backup_data.pkl", "wb") as save_file:
    pickle.dump(data_save, save_file)

## Temporary Data Loading

In [ ]:
with open("backup_data.pkl", "rb") as load_file:
    data_load = pickle.load(load_file)

purchases = data_load["purchases"]
owned_courses = data_load["owned_courses"]

## Get Dictionaries

In [ ]:
owned_courses_dict = {}
for course in owned_courses:
    owned_courses_dict[course.course_page] = course

In [ ]:
purchase_courses_dict = {}
for purchase in purchases:
    for course in purchase.courses:
        purchase_courses_dict[course.course_link] = None

In [ ]:
missing_owned_courses = set(owned_courses_dict.keys()).difference(set(purchase_courses_dict.keys()))
free_courses = [owned_courses_dict[course] for course in missing_owned_courses]
del purchase_courses_dict

In [ ]:
original_lengths = [len(purchase.courses) for purchase in purchases]

for purchase in purchases:
    purchase.courses = [course for course in purchase.courses if course.course_link in owned_courses_dict]

new_lengths = [len(purchase.courses) for purchase in purchases]

In [ ]:
for ind, purchase in enumerate(purchases):
    if original_lengths[ind] != new_lengths[ind] and new_lengths[ind] != 0:
        purchases[ind].purchase_cost = sum([course.course_cost for course in purchases[ind].courses])
del original_lengths, new_lengths

purchases = [purchase for purchase in purchases if len(purchase.courses) != 0]

In [ ]:
course_prices = {}
for purchase in purchases:
    for course in purchase.courses:
        course_prices[course.course_link] = UdemyCoursePurchaseDetails(
            course_purchase=purchase.purchase_number, 
            course_date=purchase.purchase_date, 
            course_cost=course.course_cost)

In [ ]:
for crs_ind, course in enumerate(owned_courses):
    if course.course_page in course_prices:
        owned_courses[crs_ind].purchase = course_prices[course.course_page]
del course_prices

In [ ]:
for p_ind, purchase in enumerate(purchases):
    for c_ind, course in enumerate(purchase.courses):
        purchases[p_ind].courses[c_ind] = owned_courses_dict[course.course_link]
del owned_courses_dict

## Printing

In [ ]:
purchases

In [ ]:
owned_courses